# Day 031 Project: Hardened Batch Processor

## What You're Building

A resilient batch processor that:
1. Applies `resilient_step` to every input item (retrying transient failures)
2. Routes permanently failed items to a `DeadLetterQueue`
3. Generates an AI-powered incident report via `ai_resilience_report`

Think of this as a hardened version of the Day 30 pipeline: instead of a single pipeline that stops on failure, you have a batch that processes every item and preserves the audit trail.

## Project Requirements

1. Define a `process_item(item)` function that might fail transiently
2. Process at least 5 items using `process_batch_with_dlq`
3. Store the result as `result` (the list of step result dicts)
4. Drain the DLQ into `dlq_items`
5. Generate a report with `ai_resilience_report` stored as `report`
6. Verify with `_run_project_checks()`

In [ ]:
import ollama
import time
from datetime import datetime

## Provided: All Helper Functions

In [ ]:
import ollama
import time
from datetime import datetime


import time


def retry(fn, max_attempts: int = 3, base_delay: float = 1.0, backoff: float = 2.0):
    last_error = None
    for attempt in range(max_attempts):
        try:
            return fn()
        except Exception as e:
            last_error = e
            if attempt < max_attempts - 1:
                time.sleep(base_delay * (backoff ** attempt))
    raise last_error


from datetime import datetime


class DeadLetterQueue:
    def __init__(self):
        self._items: list = []

    def add(self, item, error: str, context: dict | None = None) -> None:
        self._items.append({
            "item":     item,
            "error":    error,
            "context":  context or {},
            "added_at": datetime.now().isoformat(),
        })

    def drain(self) -> list:
        items, self._items = self._items, []
        return items

    def peek(self) -> list:
        return list(self._items)

    def size(self) -> int:
        return len(self._items)


def resilient_step(
    name: str,
    fn,
    max_attempts: int = 3,
    base_delay: float = 1.0,
    backoff: float = 2.0,
) -> dict:
    start      = time.time()
    last_error = None
    for attempt in range(max_attempts):
        try:
            result = fn()
            return {
                "name":       name,
                "status":     "ok",
                "result":     result,
                "error":      None,
                "duration_s": round(time.time() - start, 3),
                "attempts":   attempt + 1,
            }
        except Exception as e:
            last_error = e
            if attempt < max_attempts - 1:
                time.sleep(base_delay * (backoff ** attempt))
    return {
        "name":       name,
        "status":     "error",
        "result":     None,
        "error":      str(last_error),
        "duration_s": round(time.time() - start, 3),
        "attempts":   max_attempts,
    }


def process_batch_with_dlq(
    items: list,
    process_fn,
    dlq: DeadLetterQueue,
    max_attempts: int = 3,
    base_delay: float = 1.0,
    backoff: float = 2.0,
) -> list:
    results = []
    for item in items:
        r = resilient_step(
            str(item),
            lambda i=item: process_fn(i),
            max_attempts=max_attempts,
            base_delay=base_delay,
            backoff=backoff,
        )
        if r["status"] == "error":
            dlq.add(item, r["error"])
        results.append(r)
    return results


def ai_resilience_report(
    batch_results: list,
    dlq_items: list,
    model: str = "llama3.2",
) -> str:
    total        = len(batch_results)
    passed       = sum(1 for r in batch_results if r["status"] == "ok")
    avg_attempts = (
        round(sum(r.get("attempts", 1) for r in batch_results) / total, 2)
        if total else 0.0
    )
    lines = [
        f"Batch run: {passed}/{total} items succeeded, "
        f"{len(dlq_items)} failed to DLQ.",
        f"Average attempts per item: {avg_attempts}.",
    ]
    for entry in dlq_items[:3]:
        lines.append(f"  DLQ: {entry['item']} \u2014 {entry['error']}")

    response = ollama.chat(
        model=model,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a reliability engineer. "
                    "Summarise a batch processing run in 2\u20133 sentences. "
                    "Focus on the failure rate and recommend one concrete action."
                ),
            },
            {
                "role": "user",
                "content": "\n".join(lines) + "\n\nSummarise and recommend:",
            },
        ],
    )
    return response["message"]["content"]

## Your Batch Processor

Design a process_item function and a batch to process. To test resilience, you can make the function fail for certain inputs or simulate transient failures with a counter.

In [ ]:
# Example: process a list of URLs or IDs — some will fail
# Your process_item can be anything: fetch data, transform a record, call an API

_attempt_count = {}  # tracks per-item attempt count for simulated transience

def process_item(item):
    # Simulate: some items need 2 attempts, some always fail
    _attempt_count[item] = _attempt_count.get(item, 0) + 1
    if str(item).startswith('fail_'):
        raise ValueError(f'permanently invalid: {item}')
    if str(item).startswith('retry_') and _attempt_count[item] < 2:
        raise ConnectionError(f'transient error, attempt {_attempt_count[item]}')
    return f'processed: {item}'

# Create your batch
items = [
    'item_1', 'item_2', 'retry_3', 'fail_4', 'item_5',
    'retry_6', 'item_7', 'fail_8',
]

dlq = DeadLetterQueue()

result = process_batch_with_dlq(
    items, process_item, dlq,
    max_attempts=3, base_delay=0.0,  # base_delay=0.0 for fast execution
)

dlq_items = dlq.drain()

print(f'Processed {len(result)} items')
print(f'Succeeded: {sum(1 for r in result if r["status"]=="ok")}')
print(f'Failed:    {len(dlq_items)} items in DLQ')
for r in result:
    print(f'  {r["status"]:7} {r["name"]:15} attempts={r["attempts"]}')

In [ ]:
report = ai_resilience_report(result, dlq_items)
print('\nAI Incident Report:')
print(report)

## Checks

In [ ]:
def _run_project_checks():
    total = 5
    passed = 0

    # Check 1: result is a list
    try:
        assert 'result' in globals(), 'result not defined'
        assert isinstance(result, list), f'result should be list, got {type(result)}'
        passed += 1; print(f'\u2705 Check 1: result is a list of {len(result)} records')
    except Exception as e:
        print(f'\u274c Check 1: {e}')

    # Check 2: at least 5 items processed
    try:
        assert len(result) >= 5, f'expected >=5 items, got {len(result)}'
        passed += 1; print(f'\u2705 Check 2: {len(result)} items processed')
    except Exception as e:
        print(f'\u274c Check 2: {e}')

    # Check 3: dlq_items is a list
    try:
        assert 'dlq_items' in globals(), 'dlq_items not defined'
        assert isinstance(dlq_items, list), f'dlq_items should be list'
        passed += 1; print(f'\u2705 Check 3: dlq_items is list with {len(dlq_items)} entries')
    except Exception as e:
        print(f'\u274c Check 3: {e}')

    # Check 4: all results have attempts key
    try:
        for r in result:
            assert 'attempts' in r, f"result missing 'attempts' key: {list(r)}"
        passed += 1; print('\u2705 Check 4: all results have attempts key')
    except Exception as e:
        print(f'\u274c Check 4: {e}')

    # Check 5: report is a non-empty string
    try:
        assert 'report' in globals(), 'report not defined'
        assert isinstance(report, str) and len(report) > 10, \
            f'report should be non-empty str: {report!r}'
        passed += 1; print(f'\u2705 Check 5: report is {len(report)} chars')
    except Exception as e:
        print(f'\u274c Check 5: {e}')

    if passed == total:
        print('\U0001f389 Project complete!')
    print(f'\nScore: {passed}/{total}')


_run_project_checks()

## Bonus Challenges

- Add jitter to the retry backoff: `time.sleep(base_delay * (backoff**attempt) + random.uniform(0, 0.5))`
- Persist the DLQ to disk as JSON after each batch run so failures survive restarts
- Implement a `replay_dlq(dlq_json_path, process_fn)` function that re-processes items from a saved DLQ file
- Add an `exception_filter` parameter to process_batch_with_dlq that only retries specific exception types (e.g., `ConnectionError`) and immediately routes others to DLQ
- Wire Day 30's `Pipeline` with Day 31's `resilient_step` — replace `chain_steps` with a `resilient_chain_steps` variant